# 11 – Subgruppen-Performance & Responsible Use

Dieses Notebook untersucht die final festgelegten Modelle in ausgewählten Portfolio-Subgruppen.

## Ziele
- systematische Über- oder Unterschätzungen in einzelnen Subgruppen erkennen,
- den festgelegten Train-/Test-Split unverändert beibehalten,
- Holdout-Daten ausschließlich für diagnostische Analysen verwenden,
- Modellgrenzen und Regeln für einen verantwortungsvollen Einsatz dokumentieren.

In diesem Notebook erfolgen keine Modellauswahl, kein Hyperparameter-Tuning und keine erneute Aufteilung der Daten.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path("..")

CURATED_PATH = PROJECT_ROOT / "data" / "processed" / "policy_curated.pq"
SPLIT_PATH = PROJECT_ROOT / "data" / "processed" / "policy_train_test_split.pq"
MODEL_PRED_PATH = PROJECT_ROOT / "data" / "output" / "model_predictions.pq"
TWEEDIE_PRED_PATH = PROJECT_ROOT / "data" / "output" / "tweedie_predictions.pq"


curated = pd.read_parquet(CURATED_PATH)
split = pd.read_parquet(SPLIT_PATH)
model_pred = pd.read_parquet(MODEL_PRED_PATH)
tweedie_pred = pd.read_parquet(TWEEDIE_PRED_PATH)


print("Shapes:")
print("curated:", curated.shape)
print("split:", split.shape)
print("model_pred:", model_pred.shape)
print("tweedie_pred:", tweedie_pred.shape)

print("\nSplit counts:")
print(split["split"].value_counts())

print("\nDuplicate ID checks:")
print("curated:", curated["IDpol"].duplicated().sum())
print("split:", split["IDpol"].duplicated().sum())
print("model_pred:", model_pred["idpol"].duplicated().sum())
print("tweedie_pred:", tweedie_pred["idpol"].duplicated().sum())

Shapes:
curated: (678013, 25)
split: (678013, 2)
model_pred: (678013, 11)
tweedie_pred: (678013, 5)

Split counts:
split
train    542410
test     135603
Name: count, dtype: int64

Duplicate ID checks:
curated: 0
split: 0
model_pred: 0
tweedie_pred: 0


In [2]:
curated_ids = pd.Index(curated["IDpol"])
split_ids = pd.Index(split["IDpol"])
model_pred_ids = pd.Index(model_pred["idpol"])
tweedie_pred_ids = pd.Index(tweedie_pred["idpol"])

print("ID coverage checks:")
print("Curated -> Split missing:",
      len(curated_ids.difference(split_ids)))
print("Split -> Curated missing:",
      len(split_ids.difference(curated_ids)))

print("Curated -> Model predictions missing:",
      len(curated_ids.difference(model_pred_ids)))
print("Model predictions -> Curated missing:",
      len(model_pred_ids.difference(curated_ids)))

print("Curated -> Tweedie predictions missing:",
      len(curated_ids.difference(tweedie_pred_ids)))
print("Tweedie predictions -> Curated missing:",
      len(tweedie_pred_ids.difference(curated_ids)))

ID coverage checks:
Curated -> Split missing: 0
Split -> Curated missing: 0
Curated -> Model predictions missing: 0
Model predictions -> Curated missing: 0
Curated -> Tweedie predictions missing: 0
Tweedie predictions -> Curated missing: 0


In [3]:
diagnostic = (
    curated
    .merge(
        split,
        on="IDpol",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        model_pred.rename(columns={"idpol": "IDpol"}),
        on="IDpol",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        tweedie_pred.rename(columns={"idpol": "IDpol"}),
        on="IDpol",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_tweedie")
    )
)

print("Diagnostic shape:", diagnostic.shape)

print("\nSplit counts:")
print(diagnostic["split"].value_counts())

print("\nMissing prediction checks:")
print(
    diagnostic[
        [
            "predicted_frequency",
            "predicted_severity",
            "predicted_pure_premium",
            "predicted_pure_premium_tweedie"
        ]
    ]
    .isna()
    .sum()
)

Diagnostic shape: (678013, 40)

Split counts:
split
train    542410
test     135603
Name: count, dtype: int64

Missing prediction checks:
predicted_frequency               0
predicted_severity                0
predicted_pure_premium            0
predicted_pure_premium_tweedie    0
dtype: int64


In [4]:
holdout = diagnostic.loc[diagnostic["split"] == "test"].copy()

subgroup_columns = [
    "driver_age_group",
    "Region",
    "Area",
    "Density"
]

print("Holdout shape:", holdout.shape)

print("\nMissing subgroup values:")
print(holdout[subgroup_columns].isna().sum())

print("\nDriver age groups:")
print(holdout["driver_age_group"].value_counts(dropna=False))

Holdout shape: (135603, 40)

Missing subgroup values:
driver_age_group    0
Region              0
Area                0
Density             0
dtype: int64

Driver age groups:
driver_age_group
35-44    34198
45-54    32642
25-34    28212
55-64    20036
65-74    10210
18-24     5955
75+       4350
Name: count, dtype: int64


In [5]:
frequency_by_age = (
    holdout
    .groupby("driver_age_group", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_claims=("ClaimNb", "sum"),
        predicted_claims=("expected_claim_count", "sum")
    )
)

frequency_by_age["observed_frequency"] = (
    frequency_by_age["observed_claims"] / frequency_by_age["exposure"]
)

frequency_by_age["predicted_frequency"] = (
    frequency_by_age["predicted_claims"] / frequency_by_age["exposure"]
)

frequency_by_age["absolute_deviation"] = (
    frequency_by_age["predicted_frequency"]
    - frequency_by_age["observed_frequency"]
)

frequency_by_age["relative_deviation"] = (
    frequency_by_age["predicted_frequency"]
    / frequency_by_age["observed_frequency"]
    - 1
)

age_order = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65-74",
    "75+"
]

frequency_by_age = frequency_by_age.reindex(age_order)

frequency_by_age

,policy_count,exposure,observed_claims,predicted_claims,observed_frequency,predicted_frequency,absolute_deviation,relative_deviation
driver_age_group,,,,,,,,
18-24,5955,2424.396119,461,472.059853,0.190150,0.194712,0.004562,0.023991
25-34,28212,12892.544473,1253,1271.291508,0.097188,0.098607,0.001419,0.014598
35-44,34198,17522.767957,1681,1634.807587,0.095932,0.093296,-0.002636,-0.027479
45-54,32642,17912.378251,1892,1889.971105,0.105625,0.105512,-0.000113,-0.001072
55-64,20036,11316.824973,1017,1028.465523,0.089866,0.090879,0.001013,0.011274
65-74,10210,6456.489457,587,617.379890,0.090916,0.095622,0.004705,0.051754
75+,4350,3058.201412,345,304.877133,0.112811,0.099692,-0.013120,-0.116298


### Interpretation – Frequency by Driver Age

Die Frequency-Prognosen zeigen über die meisten Altersgruppen eine gute Kalibrierung. 
Für die Gruppen zwischen 18 und 64 Jahren liegen die relativen Abweichungen überwiegend nahe am beobachteten Niveau.

Die deutlichste Abweichung zeigt sich in der Gruppe **75+**, in der die beobachtete Schadenfrequenz um rund **11,6 % unterschätzt** wird. 
Für die Gruppe **65–74** liegt eine moderate Überschätzung von rund **5,2 %** vor.

Diese Ergebnisse werden ausschließlich als Holdout-Diagnostik interpretiert und führen nicht zu einer nachträglichen Modellauswahl oder Anpassung. 
Insbesondere kleinere Subgruppen sollten aufgrund höherer statistischer Unsicherheit nicht überinterpretiert werden.

In [6]:
frequency_by_region = (
    holdout
    .groupby("Region", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_claims=("ClaimNb", "sum"),
        predicted_claims=("expected_claim_count", "sum")
    )
)

frequency_by_region["observed_frequency"] = (
    frequency_by_region["observed_claims"]
    / frequency_by_region["exposure"]
)

frequency_by_region["predicted_frequency"] = (
    frequency_by_region["predicted_claims"]
    / frequency_by_region["exposure"]
)

frequency_by_region["signed_deviation"] = (
    frequency_by_region["predicted_frequency"]
    - frequency_by_region["observed_frequency"]
)

frequency_by_region["absolute_deviation"] = (
    frequency_by_region["signed_deviation"].abs()
)

frequency_by_region["relative_deviation"] = (
    frequency_by_region["predicted_frequency"]
    / frequency_by_region["observed_frequency"]
    - 1
)

frequency_by_region = frequency_by_region.sort_values(
    "relative_deviation"
)

frequency_by_region

,policy_count,exposure,observed_claims,predicted_claims,observed_frequency,predicted_frequency,signed_deviation,absolute_deviation,relative_deviation
Region,,,,,,,,,
R43,272,107.858204,13,11.204822,0.120529,0.103885,-0.016644,0.016644,-0.138091
R23,1761,631.082002,71,61.657988,0.112505,0.097702,-0.014803,0.014803,-0.131578
R74,904,472.128319,54,51.252713,0.114376,0.108557,-0.005819,0.005819,-0.050876
R11,13906,6043.668163,836,797.646899,0.138327,0.131981,-0.006346,0.006346,-0.045877
R83,1019,448.700359,38,36.277689,0.084689,0.080851,-0.003838,0.003838,-0.045324
R93,15963,7209.482614,826,792.860685,0.114571,0.109975,-0.004597,0.004597,-0.040120
R91,7211,2947.779579,318,307.821480,0.107878,0.104425,-0.003453,0.003453,-0.032008
R72,6127,2784.835028,273,264.455546,0.098031,0.094963,-0.003068,0.003068,-0.031298
R42,439,242.120951,28,27.171310,0.115645,0.112222,-0.003423,0.003423,-0.029596


### Interpretation – Frequency by Region

Die regionale Kalibrierung ist heterogener als die Kalibrierung nach Fahrer-Altersgruppen.

Für große Regionen wie **R24** und **R82** liegen beobachtete und prognostizierte Schadenfrequenz sehr nahe beieinander. Auch weitere größere Regionen zeigen nur moderate Abweichungen.

Deutlichere relative Abweichungen treten unter anderem in **R31**, **R41** und **R26** auf, wo die Schadenfrequenz überschätzt wird. Sehr hohe relative Abweichungen in kleineren Regionen wie **R21** oder **R43** sollten aufgrund der geringen Exposition und Fallzahl nicht überinterpretiert werden.

Subgruppen-Ergebnisse werden daher immer gemeinsam mit Policy-Anzahl, Exposition und beobachteter Schadenanzahl bewertet. Sie dienen als diagnostischer Hinweis auf potenziell geringere Modellzuverlässigkeit und nicht als Grundlage für eine nachträgliche Modellanpassung.

In [7]:
print(
    holdout["Area"]
    .value_counts(dropna=False)
    .sort_index()
)

Area
A    20750
B    15171
C    38278
D    30402
E    27379
F     3623
Name: count, dtype: int64


In [8]:
frequency_by_area = (
    holdout
    .groupby("Area", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_claims=("ClaimNb", "sum"),
        predicted_claims=("expected_claim_count", "sum")
    )
)

frequency_by_area["observed_frequency"] = (
    frequency_by_area["observed_claims"]
    / frequency_by_area["exposure"]
)

frequency_by_area["predicted_frequency"] = (
    frequency_by_area["predicted_claims"]
    / frequency_by_area["exposure"]
)

frequency_by_area["signed_deviation"] = (
    frequency_by_area["predicted_frequency"]
    - frequency_by_area["observed_frequency"]
)

frequency_by_area["absolute_deviation"] = (
    frequency_by_area["signed_deviation"].abs()
)

frequency_by_area["relative_deviation"] = (
    frequency_by_area["predicted_frequency"]
    / frequency_by_area["observed_frequency"]
    - 1
)

frequency_by_area = frequency_by_area.sort_index()

frequency_by_area

,policy_count,exposure,observed_claims,predicted_claims,observed_frequency,predicted_frequency,signed_deviation,absolute_deviation,relative_deviation
Area,,,,,,,,,
A,20750,12374.745550,981,1017.308296,0.079274,0.082208,0.002934,0.002934,0.037012
B,15171,8617.476527,742,755.175395,0.086104,0.087633,0.001529,0.001529,0.017757
C,38278,20879.503035,2006,1990.929244,0.096075,0.095353,-0.000722,0.000722,-0.007513
D,30402,15426.127482,1717,1659.081396,0.111305,0.107550,-0.003755,0.003755,-0.033732
E,27379,12658.260859,1567,1557.537009,0.123793,0.123045,-0.000748,0.000748,-0.006039
F,3623,1627.489188,223,238.821261,0.137021,0.146742,0.009721,0.009721,0.070947


### Interpretation – Frequency by Area

Die Frequency-Prognosen zeigen über die Area-Kategorien insgesamt eine gute Kalibrierung. 
Für die Kategorien **A bis E** liegen die relativen Abweichungen zwischen beobachteter und prognostizierter Schadenfrequenz in einem vergleichsweise engen Bereich.

Gleichzeitig bildet das Modell den steigenden Risikoverlauf von **Area A bis F** grundsätzlich ab.

Die größte Abweichung zeigt sich in **Area F** mit einer Überschätzung von rund **7,1 %**. 
Da diese Kategorie im Holdout die kleinste Policy-Anzahl und Exposition aufweist, sollte diese Abweichung mit entsprechender Vorsicht interpretiert werden.

Insgesamt ergeben sich auf Area-Ebene keine Hinweise auf eine ausgeprägte systematische Fehlkalibrierung.

In [9]:
print("Unique Density values:", holdout["Density"].nunique())

print("\nDensity summary:")
print(
    holdout["Density"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

Unique Density values: 1583

Density summary:
count    135603.000000
mean       1792.514185
std        3965.280002
min           1.000000
10%          33.000000
25%          91.000000
50%         393.000000
75%        1662.000000
90%        4128.000000
max       27000.000000
Name: Density, dtype: float64


In [10]:
density_labels = [
    "Q1 - Very Low",
    "Q2 - Low",
    "Q3 - Medium",
    "Q4 - High",
    "Q5 - Very High"
]

holdout["density_group"], density_bins = pd.qcut(
    holdout["Density"],
    q=5,
    labels=density_labels,
    retbins=True,
    duplicates="drop"
)

print("Density bin boundaries:")
print(density_bins)

print("\nPolicies by density group:")
print(
    holdout["density_group"]
    .value_counts()
    .sort_index()
)

Density bin boundaries:
[1.00e+00 6.70e+01 2.15e+02 7.16e+02 2.71e+03 2.70e+04]

Policies by density group:
density_group
Q1 - Very Low     27427
Q2 - Low          26845
Q3 - Medium       27142
Q4 - High         27073
Q5 - Very High    27116
Name: count, dtype: int64


In [11]:
frequency_by_density = (
    holdout
    .groupby("density_group", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_claims=("ClaimNb", "sum"),
        predicted_claims=("expected_claim_count", "sum")
    )
)

frequency_by_density["observed_frequency"] = (
    frequency_by_density["observed_claims"]
    / frequency_by_density["exposure"]
)

frequency_by_density["predicted_frequency"] = (
    frequency_by_density["predicted_claims"]
    / frequency_by_density["exposure"]
)

frequency_by_density["signed_deviation"] = (
    frequency_by_density["predicted_frequency"]
    - frequency_by_density["observed_frequency"]
)

frequency_by_density["absolute_deviation"] = (
    frequency_by_density["signed_deviation"].abs()
)

frequency_by_density["relative_deviation"] = (
    frequency_by_density["predicted_frequency"]
    / frequency_by_density["observed_frequency"]
    - 1
)

frequency_by_density

,policy_count,exposure,observed_claims,predicted_claims,observed_frequency,predicted_frequency,signed_deviation,absolute_deviation,relative_deviation
density_group,,,,,,,,,
Q1 - Very Low,27427,16236.989467,1311,1348.724995,0.080742,0.083065,0.002323,0.002323,0.028776
Q2 - Low,26845,14891.796620,1361,1357.877128,0.091393,0.091183,-0.000210,0.000210,-0.002295
Q3 - Medium,27142,14361.160110,1413,1430.054476,0.098390,0.099578,0.001188,0.001188,0.012070
Q4 - High,27073,13637.052434,1587,1501.847039,0.116374,0.110130,-0.006244,0.006244,-0.053657
Q5 - Very High,27116,12456.604010,1564,1580.348962,0.125556,0.126868,0.001312,0.001312,0.010453


### Interpretation – Frequency by Density

Mit zunehmender Bevölkerungsdichte steigt die beobachtete Schadenfrequenz deutlich an. 
Das Frequency-Modell bildet diesen Risikoverlauf über die fünf quantilbasierten Dichtegruppen grundsätzlich gut ab.

In vier der fünf Gruppen liegen die relativen Abweichungen zwischen beobachteter und prognostizierter Schadenfrequenz nahe am beobachteten Niveau. 
Die deutlichste Abweichung zeigt sich in **Q4 – High**, wo die Schadenfrequenz um rund **5,4 % unterschätzt** wird.

Da die Dichtegruppen über Quantile gebildet wurden und jeweils eine vergleichbare Anzahl von Policen enthalten, ist diese Abweichung als diagnostischer Hinweis relevant. Sie stellt jedoch keine Grundlage für eine nachträgliche Modellanpassung dar.

Insgesamt zeigt das Frequency-Modell auch entlang der Dichteverteilung eine stabile Subgruppen-Kalibrierung.

In [12]:
positive_claim_holdout = holdout.loc[
    holdout["ClaimNb"] > 0
].copy()

print("Policies with ClaimNb > 0:",
      len(positive_claim_holdout))

print("\nSeverity-related DQ flags:")
print(
    positive_claim_holdout[
        ["missing_severity", "claim_count_mismatch"]
    ]
    .value_counts(dropna=False)
)

print("\nPositive total claim amount:")
print(
    (positive_claim_holdout["total_claim_amount"] > 0)
    .value_counts(dropna=False)
)

Policies with ClaimNb > 0: 6812

Severity-related DQ flags:
missing_severity  claim_count_mismatch
False             False                   4999
True              False                   1813
Name: count, dtype: int64

Positive total claim amount:
total_claim_amount
True     4999
False    1813
Name: count, dtype: int64


In [13]:
severity_holdout = holdout.loc[
    (holdout["ClaimNb"] > 0)
    & (~holdout["missing_severity"])
    & (~holdout["claim_count_mismatch"])
    & (holdout["total_claim_amount"] > 0)
].copy()

print("Severity holdout policies:", len(severity_holdout))
print("Severity holdout claims:", severity_holdout["ClaimNb"].sum())
print("Total observed claim amount:",
      severity_holdout["total_claim_amount"].sum())

print("\nObserved severity check:")
print(
    severity_holdout["total_claim_amount"].sum()
    / severity_holdout["ClaimNb"].sum()
)

Severity holdout policies: 4999
Severity holdout claims: 5312
Total observed claim amount: 11055690.67

Observed severity check:
2081.267068900602


In [14]:
severity_holdout["predicted_claim_amount"] = (
    severity_holdout["predicted_severity"]
    * severity_holdout["ClaimNb"]
)

severity_by_age = (
    severity_holdout
    .groupby("driver_age_group", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        claim_count=("ClaimNb", "sum"),
        observed_claim_amount=("total_claim_amount", "sum"),
        predicted_claim_amount=("predicted_claim_amount", "sum")
    )
)

severity_by_age["observed_severity"] = (
    severity_by_age["observed_claim_amount"]
    / severity_by_age["claim_count"]
)

severity_by_age["predicted_severity"] = (
    severity_by_age["predicted_claim_amount"]
    / severity_by_age["claim_count"]
)

severity_by_age["signed_deviation"] = (
    severity_by_age["predicted_severity"]
    - severity_by_age["observed_severity"]
)

severity_by_age["absolute_deviation"] = (
    severity_by_age["signed_deviation"].abs()
)

severity_by_age["relative_deviation"] = (
    severity_by_age["predicted_severity"]
    / severity_by_age["observed_severity"]
    - 1
)

severity_by_age = severity_by_age.reindex(age_order)

severity_by_age

,policy_count,claim_count,observed_claim_amount,predicted_claim_amount,observed_severity,predicted_severity,signed_deviation,absolute_deviation,relative_deviation
driver_age_group,,,,,,,,,
18-24,368,399,913463.92,1.592347e+06,2289.383258,3990.845836,1701.462577,1701.462577,0.743197
25-34,915,967,1831658.67,2.138310e+06,1894.166153,2211.282417,317.116264,317.116264,0.167417
35-44,1186,1257,2463977.82,2.409006e+06,1960.205107,1916.472808,-43.732300,43.732300,-0.022310
45-54,1273,1361,3068310.67,2.565626e+06,2254.453101,1885.103745,-369.349355,369.349355,-0.163831
55-64,696,728,1082179.45,1.423167e+06,1486.510234,1954.899259,468.389025,468.389025,0.315093
65-74,365,378,762398.27,8.139880e+05,2016.926640,2153.407502,136.480862,136.480862,0.067668
75+,196,222,933701.87,5.121381e+05,4205.864279,2306.928372,-1898.935908,1898.935908,-0.451497


### Interpretation – Severity by Driver Age

Die Severity-Prognosen zeigen deutlich stärkere Unterschiede zwischen den Fahrer-Altersgruppen als die Frequency-Prognosen.

Besonders ausgeprägte Abweichungen treten in den Gruppen **18–24**, **55–64** und **75+** auf. Für 18–24 wird die durchschnittliche Schadenhöhe deutlich überschätzt, während sie in der Gruppe 75+ deutlich unterschätzt wird.

Diese Ergebnisse müssen aufgrund der kleineren Anzahl beobachteter Schäden und der stark rechtsschiefen, heavy-tailed Schadenhöhenverteilung vorsichtig interpretiert werden. Einzelne Großschäden können die beobachtete Severity innerhalb kleinerer Subgruppen erheblich beeinflussen.

Die Subgruppenanalyse bestätigt damit die bereits bekannte Einschränkung des finalen Severity-Modells: Es eignet sich primär zur monetären Dekomposition im Two-Stage-Ansatz, jedoch nicht als starker eigenständiger Ranking-Score.

Die Holdout-Ergebnisse dienen ausschließlich der finalen Diagnostik und führen zu keiner nachträglichen Modellauswahl oder Modellanpassung.

In [15]:
severity_region_counts = (
    severity_holdout
    .groupby("Region", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        claim_count=("ClaimNb", "sum")
    )
    .sort_values("policy_count")
)

severity_region_counts

,policy_count,claim_count
Region,,
R43,10,10
R21,13,14
R42,20,23
R94,22,27
R83,27,30
R74,40,45
R23,45,50
R26,56,59
R22,57,58


In [16]:
severity_by_region = (
    severity_holdout
    .groupby("Region", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        claim_count=("ClaimNb", "sum"),
        observed_claim_amount=("total_claim_amount", "sum"),
        predicted_claim_amount=("predicted_claim_amount", "sum")
    )
)

severity_by_region["observed_severity"] = (
    severity_by_region["observed_claim_amount"]
    / severity_by_region["claim_count"]
)

severity_by_region["predicted_severity"] = (
    severity_by_region["predicted_claim_amount"]
    / severity_by_region["claim_count"]
)

severity_by_region["signed_deviation"] = (
    severity_by_region["predicted_severity"]
    - severity_by_region["observed_severity"]
)

severity_by_region["absolute_deviation"] = (
    severity_by_region["signed_deviation"].abs()
)

severity_by_region["relative_deviation"] = (
    severity_by_region["predicted_severity"]
    / severity_by_region["observed_severity"]
    - 1
)

severity_by_region = severity_by_region.sort_values(
    "relative_deviation"
)

severity_by_region

,policy_count,claim_count,observed_claim_amount,predicted_claim_amount,observed_severity,predicted_severity,signed_deviation,absolute_deviation,relative_deviation
Region,,,,,,,,,
R21,13,14,394925.13,2.857847e+04,28208.937857,2041.319292,-26167.618565,26167.618565,-0.927636
R93,591,651,2013046.04,1.352286e+06,3092.236621,2077.243943,-1014.992678,1014.992678,-0.328239
R94,22,27,57831.82,5.738886e+04,2141.919259,2125.513351,-16.405908,16.405908,-0.007659
R25,86,87,185034.80,1.916876e+05,2126.836782,2203.305470,76.468689,76.468689,0.035954
R24,1266,1313,2855437.00,2.960462e+06,2174.742574,2254.731346,79.988772,79.988772,0.036781
R82,795,835,1646033.89,1.766329e+06,1971.298072,2115.364378,144.066306,144.066306,0.073082
R26,56,59,115427.16,1.305124e+05,1956.392542,2212.073793,255.681251,255.681251,0.130690
R41,78,79,141914.77,1.608304e+05,1796.389494,2035.828237,239.438744,239.438744,0.133289
R74,40,45,79980.20,9.262296e+04,1777.337778,2058.287963,280.950185,280.950185,0.158074


### Interpretation – Severity by Region

Die regionale Severity-Performance ist deutlich heterogener als die Frequency-Performance.

Sehr extreme relative Abweichungen in kleinen Regionen, beispielsweise **R21**, dürfen aufgrund der geringen Schadenanzahl und der heavy-tailed Schadenhöhenverteilung nicht isoliert interpretiert werden. Einzelne Großschäden können die beobachtete durchschnittliche Schadenhöhe in solchen Gruppen stark beeinflussen.

Auch in größeren Regionen bestehen jedoch relevante Unterschiede. Während beispielsweise **R24** und **R82** eine vergleichsweise gute Kalibrierung zeigen, wird die beobachtete Severity in **R93** deutlich unterschätzt. In mehreren weiteren Regionen, darunter **R52**, **R72** und **R91**, zeigt sich eine deutliche Überschätzung.

Die Ergebnisse bestätigen, dass eine gute Kalibrierung auf Gesamtportfolio-Ebene keine gleichmäßige Subgruppen-Performance garantiert. Dies unterstützt die bereits dokumentierte Einschränkung des Severity-Modells: Es wird primär zur monetären Dekomposition im Two-Stage-Ansatz eingesetzt und nicht als eigenständiger starker Ranking-Score.

Subgruppen mit geringer Schadenanzahl werden nicht überinterpretiert. Die Analyse dient ausschließlich der finalen Holdout-Diagnostik und nicht der nachträglichen Modelloptimierung.

In [17]:
severity_by_area = (
    severity_holdout
    .groupby("Area", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        claim_count=("ClaimNb", "sum"),
        observed_claim_amount=("total_claim_amount", "sum"),
        predicted_claim_amount=("predicted_claim_amount", "sum")
    )
)

severity_by_area["observed_severity"] = (
    severity_by_area["observed_claim_amount"]
    / severity_by_area["claim_count"]
)

severity_by_area["predicted_severity"] = (
    severity_by_area["predicted_claim_amount"]
    / severity_by_area["claim_count"]
)

severity_by_area["signed_deviation"] = (
    severity_by_area["predicted_severity"]
    - severity_by_area["observed_severity"]
)

severity_by_area["absolute_deviation"] = (
    severity_by_area["signed_deviation"].abs()
)

severity_by_area["relative_deviation"] = (
    severity_by_area["predicted_severity"]
    / severity_by_area["observed_severity"]
    - 1
)

severity_by_area = severity_by_area.sort_index()

severity_by_area

,policy_count,claim_count,observed_claim_amount,predicted_claim_amount,observed_severity,predicted_severity,signed_deviation,absolute_deviation,relative_deviation
Area,,,,,,,,,
A,629,661,1873468.46,1.437150e+06,2834.294191,2174.205212,-660.088979,660.088979,-0.232894
B,489,504,782745.25,1.114541e+06,1553.065972,2211.391613,658.325641,658.325641,0.423888
C,1368,1444,2865680.76,3.105542e+06,1984.543463,2150.652170,166.108707,166.108707,0.083701
D,1227,1324,2825625.72,2.873710e+06,2134.158399,2170.475575,36.317176,36.317176,0.017017
E,1147,1228,2523623.38,2.617914e+06,2055.067899,2131.851457,76.783558,76.783558,0.037363
F,139,151,184547.10,3.057269e+05,1222.166225,2024.681425,802.515200,802.515200,0.656633


### Interpretation – Severity by Area

Die Severity-Prognosen zeigen auch auf Area-Ebene eine deutlich heterogenere Kalibrierung als die Frequency-Prognosen.

Während die Kategorien **D** und **E** eine vergleichsweise gute Übereinstimmung zwischen beobachteter und prognostizierter Schadenhöhe zeigen, treten in **A**, **B** und insbesondere **F** deutlichere Abweichungen auf.

In **Area A** wird die beobachtete Severity um rund **23 % unterschätzt**, während sie in **Area B** um rund **42 % überschätzt** wird. Die stärkste relative Überschätzung zeigt sich in **Area F** mit rund **66 %**. Da Area F jedoch deutlich weniger beobachtete Schäden enthält, sollte dieses Ergebnis vorsichtig interpretiert werden.

Die Ergebnisse bestätigen erneut, dass das Severity-Modell auf Subgruppenebene nur eingeschränkt stabil ist. Diese Einschränkung ist konsistent mit der bereits dokumentierten schwachen Ranking-Performance des finalen Severity-Modells.

In [18]:
severity_by_density = (
    severity_holdout
    .groupby("density_group", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        claim_count=("ClaimNb", "sum"),
        observed_claim_amount=("total_claim_amount", "sum"),
        predicted_claim_amount=("predicted_claim_amount", "sum")
    )
)

severity_by_density["observed_severity"] = (
    severity_by_density["observed_claim_amount"]
    / severity_by_density["claim_count"]
)

severity_by_density["predicted_severity"] = (
    severity_by_density["predicted_claim_amount"]
    / severity_by_density["claim_count"]
)

severity_by_density["signed_deviation"] = (
    severity_by_density["predicted_severity"]
    - severity_by_density["observed_severity"]
)

severity_by_density["absolute_deviation"] = (
    severity_by_density["signed_deviation"].abs()
)

severity_by_density["relative_deviation"] = (
    severity_by_density["predicted_severity"]
    / severity_by_density["observed_severity"]
    - 1
)

severity_by_density

,policy_count,claim_count,observed_claim_amount,predicted_claim_amount,observed_severity,predicted_severity,signed_deviation,absolute_deviation,relative_deviation
density_group,,,,,,,,,
Q1 - Very Low,851,891,2242381.13,1.954904e+06,2516.701605,2194.056368,-322.645236,322.645236,-0.128202
Q2 - Low,925,970,1747671.68,2.089144e+06,1801.723381,2153.756217,352.032836,352.032836,0.195387
Q3 - Medium,965,1017,1987959.00,2.198591e+06,1954.728614,2161.839857,207.111243,207.111243,0.105954
Q4 - High,1141,1236,2768873.65,2.681361e+06,2240.189037,2169.386157,-70.802880,70.802880,-0.031606
Q5 - Very High,1117,1198,2308805.21,2.530583e+06,1927.216369,2112.339497,185.123128,185.123128,0.096057


### Interpretation – Severity by Density

Auch entlang der Dichteverteilung zeigt das Severity-Modell eine heterogene Subgruppen-Performance.

Die stärksten relativen Abweichungen treten in **Q1 – Very Low** mit einer Unterschätzung von rund **12,8 %** und in **Q2 – Low** mit einer Überschätzung von rund **19,5 %** auf. Die Gruppe **Q4 – High** zeigt dagegen mit rund **−3,2 %** eine vergleichsweise gute Kalibrierung.

Im Gegensatz zu einzelnen regionalen Subgruppen enthalten die quantilbasierten Dichtegruppen jeweils eine vergleichbare und ausreichend große Anzahl beobachteter Schäden. Die Unterschiede sind daher als relevante diagnostische Hinweise zu betrachten.

Aufgrund der heavy-tailed Schadenhöhenverteilung bleiben Severity-Ergebnisse jedoch grundsätzlich schwankungsanfälliger als Frequency-Ergebnisse.

Insgesamt bestätigt auch diese Analyse die begrenzte Subgruppenstabilität des finalen Severity-Modells und unterstützt seine Verwendung primär zur monetären Dekomposition anstelle eines eigenständigen Ranking-Scores.

In [19]:
loss_holdout = holdout.loc[
    ~holdout["missing_severity"]
].copy()

print("Common observable holdout policies:", len(loss_holdout))

print("\nClaim status:")
print(
    (loss_holdout["ClaimNb"] > 0)
    .value_counts()
)

print("\nMissing observed claim amount:")
print(
    loss_holdout["total_claim_amount"]
    .isna()
    .sum()
)

print("\nExposure consistency:")
print(
    (loss_holdout["Exposure"] - loss_holdout["exposure_used"]).abs().describe()
)

Common observable holdout policies: 133790

Claim status:
ClaimNb
False    128791
True       4999
Name: count, dtype: int64

Missing observed claim amount:
0

Exposure consistency:
count    133790.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64


In [20]:
tweedie_by_age = (
    loss_holdout
    .groupby("driver_age_group", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_loss=("total_claim_amount", "sum"),
        predicted_loss=("expected_loss_tweedie", "sum")
    )
)

tweedie_by_age["observed_pure_premium"] = (
    tweedie_by_age["observed_loss"]
    / tweedie_by_age["exposure"]
)

tweedie_by_age["predicted_pure_premium"] = (
    tweedie_by_age["predicted_loss"]
    / tweedie_by_age["exposure"]
)

tweedie_by_age["signed_deviation"] = (
    tweedie_by_age["predicted_pure_premium"]
    - tweedie_by_age["observed_pure_premium"]
)

tweedie_by_age["absolute_deviation"] = (
    tweedie_by_age["signed_deviation"].abs()
)

tweedie_by_age["relative_deviation"] = (
    tweedie_by_age["predicted_pure_premium"]
    / tweedie_by_age["observed_pure_premium"]
    - 1
)

tweedie_by_age = tweedie_by_age.reindex(age_order)

tweedie_by_age

,policy_count,exposure,observed_loss,predicted_loss,observed_pure_premium,predicted_pure_premium,signed_deviation,absolute_deviation,relative_deviation
driver_age_group,,,,,,,,,
18-24,5898,2407.109725,913463.92,1.956766e+06,379.485783,812.911084,433.425301,433.425301,1.142138
25-34,27943,12794.921740,1831658.67,1.906749e+06,143.155129,149.023876,5.868748,5.868748,0.040996
35-44,33801,17350.354296,2463977.82,2.431346e+06,142.013113,140.132332,-1.880781,1.880781,-0.013244
45-54,32140,17654.294536,3068310.67,2.675197e+06,173.799676,151.532372,-22.267304,22.267304,-0.128121
55-64,19761,11169.064044,1082179.45,1.441094e+06,96.890791,129.025518,32.134727,32.134727,0.331659
65-74,10008,6323.849457,762398.27,8.848424e+05,120.559206,139.921489,19.362283,19.362283,0.160604
75+,4239,2964.911412,933701.87,4.898919e+05,314.917291,165.229853,-149.687438,149.687438,-0.475323


### Interpretation – Tweedie Pure Premium by Driver Age

Die Tweedie-Pure-Premium-Prognosen zeigen deutliche Unterschiede zwischen den Fahrer-Altersgruppen.

Für die Gruppen **25–34** und **35–44** liegen beobachtetes und prognostiziertes Pure Premium vergleichsweise nahe beieinander. Deutlich stärkere Abweichungen zeigen sich dagegen in mehreren anderen Altersgruppen.

Besonders auffällig ist die Gruppe **18–24**, in der das beobachtete Pure Premium deutlich überschätzt wird. Auch für **55–64** und **65–74** zeigt sich eine Überschätzung. In der Gruppe **75+** wird das beobachtete Pure Premium dagegen deutlich unterschätzt.

Die Ergebnisse zeigen, dass eine gute Portfolio-Performance und eine nützliche Ranking-Fähigkeit keine gleichmäßige Kalibrierung in allen Subgruppen garantieren.

Das Tweedie-Modell wird daher weiterhin als direkter Pure-Premium-Risikoscore für Segmentierung und Entscheidungsunterstützung verwendet, jedoch nicht als automatischer Underwriting-Entscheider oder als unmittelbar zu verwendender Kundenpreis.

Die Analyse dient ausschließlich der finalen Holdout-Diagnostik und führt zu keiner nachträglichen Modellanpassung.

In [21]:
tweedie_region_counts = (
    loss_holdout
    .groupby("Region", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        claim_policies=("ClaimNb", lambda x: (x > 0).sum()),
        observed_loss=("total_claim_amount", "sum")
    )
    .sort_values("policy_count")
)

tweedie_region_counts

,policy_count,exposure,claim_policies,observed_loss
Region,,,,
R43,269,106.698204,10,14300.38
R42,434,241.290951,20,25743.77
R21,597,242.631918,13,394925.13
R94,869,344.932869,22,57831.82
R74,896,470.048319,40,79980.20
R83,1012,447.050359,27,38050.72
R22,1568,703.348510,57,108936.32
R23,1742,623.982002,45,84811.26
R26,2081,997.969454,56,115427.16


In [22]:
tweedie_by_region = (
    loss_holdout
    .groupby("Region", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_loss=("total_claim_amount", "sum"),
        predicted_loss=("expected_loss_tweedie", "sum")
    )
)

tweedie_by_region["observed_pure_premium"] = (
    tweedie_by_region["observed_loss"]
    / tweedie_by_region["exposure"]
)

tweedie_by_region["predicted_pure_premium"] = (
    tweedie_by_region["predicted_loss"]
    / tweedie_by_region["exposure"]
)

tweedie_by_region["signed_deviation"] = (
    tweedie_by_region["predicted_pure_premium"]
    - tweedie_by_region["observed_pure_premium"]
)

tweedie_by_region["absolute_deviation"] = (
    tweedie_by_region["signed_deviation"].abs()
)

tweedie_by_region["relative_deviation"] = (
    tweedie_by_region["predicted_pure_premium"]
    / tweedie_by_region["observed_pure_premium"]
    - 1
)

tweedie_by_region = tweedie_by_region.sort_values(
    "relative_deviation"
)

tweedie_by_region

,policy_count,exposure,observed_loss,predicted_loss,observed_pure_premium,predicted_pure_premium,signed_deviation,absolute_deviation,relative_deviation
Region,,,,,,,,,
R21,597,242.631918,394925.13,1.990408e+05,1627.671798,820.340372,-807.331426,807.331426,-0.496004
R74,896,470.048319,79980.20,5.755826e+04,170.153146,122.451799,-47.701346,47.701346,-0.280344
R93,15795,7159.334417,2013046.04,1.641801e+06,281.177819,229.323174,-51.854645,51.854645,-0.184419
R23,1742,623.982002,84811.26,8.388261e+04,135.919401,134.431142,-1.488259,1.488259,-0.010950
R83,1012,447.050359,38050.72,3.938708e+04,85.115064,88.104348,2.989284,2.989284,0.035121
R24,31784,20203.032505,2855437.00,3.024953e+06,141.337049,149.727649,8.390600,8.390600,0.059366
R82,16669,8899.982091,1646033.89,1.775086e+06,184.948000,199.448281,14.500281,14.500281,0.078402
R11,13612,5948.879037,948342.34,1.032070e+06,159.415301,173.489809,14.074508,14.074508,0.088288
R42,434,241.290951,25743.77,2.818047e+04,106.691817,116.790420,10.098603,10.098603,0.094652


### Interpretation – Tweedie Pure Premium by Region

Die regionale Kalibrierung des Tweedie-Pure-Premium-Modells ist heterogen.

In großen Regionen wie **R24**, **R82** und **R11** liegen beobachtetes und prognostiziertes Pure Premium vergleichsweise nahe beieinander. In anderen ausreichend großen Regionen treten jedoch deutlichere Abweichungen auf. Besonders **R93** zeigt eine relevante Unterschätzung, während unter anderem **R52**, **R72**, **R91** und **R53** eine stärkere Überschätzung aufweisen.

Extremere relative Abweichungen in kleineren Regionen, beispielsweise **R21**, werden aufgrund der geringen Anzahl beobachteter Schadenfälle und der heavy-tailed Schadenverteilung nicht überinterpretiert.

Die Ergebnisse zeigen, dass die gute Gesamtportfolio-Performance des Tweedie-Modells keine gleichmäßige Kalibrierung in allen regionalen Subgruppen garantiert. Der Score eignet sich daher für Risikosegmentierung und Entscheidungsunterstützung, jedoch nicht für automatische Entscheidungen oder als direkt anzuwendender Kundenpreis.

Die Analyse dient ausschließlich der finalen Holdout-Diagnostik und führt zu keiner nachträglichen Modellanpassung.

In [23]:
tweedie_by_area = (
    loss_holdout
    .groupby("Area", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_loss=("total_claim_amount", "sum"),
        predicted_loss=("expected_loss_tweedie", "sum")
    )
)

tweedie_by_area["observed_pure_premium"] = (
    tweedie_by_area["observed_loss"]
    / tweedie_by_area["exposure"]
)

tweedie_by_area["predicted_pure_premium"] = (
    tweedie_by_area["predicted_loss"]
    / tweedie_by_area["exposure"]
)

tweedie_by_area["signed_deviation"] = (
    tweedie_by_area["predicted_pure_premium"]
    - tweedie_by_area["observed_pure_premium"]
)

tweedie_by_area["absolute_deviation"] = (
    tweedie_by_area["signed_deviation"].abs()
)

tweedie_by_area["relative_deviation"] = (
    tweedie_by_area["predicted_pure_premium"]
    / tweedie_by_area["observed_pure_premium"]
    - 1
)

tweedie_by_area = tweedie_by_area.sort_index()

tweedie_by_area

,policy_count,exposure,observed_loss,predicted_loss,observed_pure_premium,predicted_pure_premium,signed_deviation,absolute_deviation,relative_deviation
Area,,,,,,,,,
A,20452,12188.193692,1873468.46,1.492072e+06,153.711740,122.419462,-31.292278,31.292278,-0.203578
B,14943,8484.141062,782745.25,1.173206e+06,92.259811,138.282286,46.022475,46.022475,0.498836
C,37749,20585.981177,2865680.76,3.138593e+06,139.205449,152.462659,13.257210,13.257210,0.095235
D,30032,15265.268356,2825625.72,2.911940e+06,185.101608,190.755872,5.654264,5.654264,0.030547
E,27060,12533.264466,2523623.38,2.750232e+06,201.354036,219.434635,18.080599,18.080599,0.089795
F,3554,1607.656456,184547.10,3.198425e+05,114.792622,198.949521,84.156899,84.156899,0.733121


### Interpretation – Tweedie Pure Premium by Area

Die Tweedie-Pure-Premium-Prognosen zeigen auch auf Area-Ebene eine heterogene Kalibrierung.

Die Kategorien **C**, **D** und **E** weisen eine vergleichsweise gute Übereinstimmung zwischen beobachtetem und prognostiziertem Pure Premium auf. Besonders **Area D** zeigt nur eine geringe relative Abweichung.

Deutlichere Unterschiede zeigen sich dagegen in **Area A** mit einer Unterschätzung von rund **20 %** und in **Area B** mit einer Überschätzung von rund **50 %**. Die größte relative Überschätzung tritt in **Area F** auf. Da Area F jedoch die kleinste Gruppe darstellt, sollte dieses Ergebnis vorsichtiger interpretiert werden.

Insgesamt bestätigt auch die Area-Analyse, dass das Tweedie-Modell auf Portfolioebene und für die Risikosegmentierung nützlich ist, die Kalibrierung jedoch nicht in allen Subgruppen gleich stabil ausfällt.

In [24]:
tweedie_by_density = (
    loss_holdout
    .groupby("density_group", observed=True)
    .agg(
        policy_count=("IDpol", "count"),
        exposure=("Exposure", "sum"),
        observed_loss=("total_claim_amount", "sum"),
        predicted_loss=("expected_loss_tweedie", "sum")
    )
)

tweedie_by_density["observed_pure_premium"] = (
    tweedie_by_density["observed_loss"]
    / tweedie_by_density["exposure"]
)

tweedie_by_density["predicted_pure_premium"] = (
    tweedie_by_density["predicted_loss"]
    / tweedie_by_density["exposure"]
)

tweedie_by_density["signed_deviation"] = (
    tweedie_by_density["predicted_pure_premium"]
    - tweedie_by_density["observed_pure_premium"]
)

tweedie_by_density["absolute_deviation"] = (
    tweedie_by_density["signed_deviation"].abs()
)

tweedie_by_density["relative_deviation"] = (
    tweedie_by_density["predicted_pure_premium"]
    / tweedie_by_density["observed_pure_premium"]
    - 1
)

tweedie_by_density

,policy_count,exposure,observed_loss,predicted_loss,observed_pure_premium,predicted_pure_premium,signed_deviation,absolute_deviation,relative_deviation
density_group,,,,,,,,,
Q1 - Very Low,27031,15992.962145,2242381.13,2.008340e+06,140.210494,125.576494,-14.634000,14.634000,-0.104372
Q2 - Low,26477,14681.766620,1747671.68,2.085569e+06,119.036879,142.051609,23.014729,23.014729,0.193341
Q3 - Medium,26767,14161.895520,1987959.00,2.332419e+06,140.373794,164.696783,24.322990,24.322990,0.173273
Q4 - High,26746,13499.376040,2768873.65,2.657727e+06,205.111232,196.877771,-8.233461,8.233461,-0.040141
Q5 - Very High,26769,12328.504884,2308805.21,2.701832e+06,187.273739,219.153252,31.879512,31.879512,0.170229


### Interpretation – Tweedie Pure Premium by Density

Die Tweedie-Pure-Premium-Prognosen zeigen auch entlang der Dichteverteilung eine heterogene Kalibrierung.

Die Gruppe **Q4 – High** weist mit einer relativen Abweichung von rund **−4 %** die beste Übereinstimmung zwischen beobachtetem und prognostiziertem Pure Premium auf.

In **Q2 – Low**, **Q3 – Medium** und **Q5 – Very High** wird das beobachtete Pure Premium dagegen um rund **17–19 % überschätzt**, während **Q1 – Very Low** eine Unterschätzung von rund **10 %** zeigt.

Da die Dichtegruppen quantilbasiert gebildet wurden und jeweils eine vergleichbare Anzahl von Policen enthalten, sind diese Unterschiede als relevante diagnostische Hinweise zu betrachten.

Die Ergebnisse bestätigen, dass das Tweedie-Modell für Portfolio-Ranking und Risikosegmentierung nützlich ist, die Kalibrierung jedoch nicht in allen Subgruppen gleich stabil ausfällt.

## Responsible Use und Gesamtfazit

### Gesamtfazit der Subgruppenanalyse

Die finalen Modelle zeigen auf Gesamtportfolio-Ebene eine grundsätzlich brauchbare Performance, die Subgruppenanalyse macht jedoch deutlich, dass diese Leistung nicht in allen Gruppen gleichmäßig ist.

Das **Frequency-Modell** weist insgesamt die stabilste Subgruppen-Kalibrierung auf. Über Fahrer-Altersgruppen, Area und Dichte bleiben die Abweichungen überwiegend moderat. Einzelne Auffälligkeiten, beispielsweise in der Altersgruppe 75+ oder bestimmten Regionen, sollten im Monitoring weiter beobachtet werden.

Das **Severity-Modell** zeigt deutlich stärkere Schwankungen zwischen den Subgruppen. Dies ist konsistent mit der bereits bekannten schwachen Ranking-Performance und der heavy-tailed Schadenhöhenverteilung. Das Modell wird daher primär zur monetären Dekomposition im Two-Stage-Ansatz verwendet und nicht als eigenständiger starker Risikoscore.

Das **Tweedie-Modell** liefert einen nützlichen direkten Pure-Premium-Risikoscore für Ranking und Segmentierung. Gleichzeitig zeigen die Holdout-Diagnosen relevante Kalibrierungsunterschiede zwischen einzelnen Alters-, Regional-, Area- und Dichtegruppen. Eine gute Gesamtportfolio-Performance garantiert daher keine gleichmäßige Performance in jeder Subgruppe.

### Responsible Use

Die Modellergebnisse dienen als **Entscheidungsunterstützung** und nicht als vollständig automatisierte Entscheidungslogik.

Insbesondere gelten folgende Grenzen:

- Ein **HIGH-Risk-Segment bedeutet keine automatische Ablehnung** einer Police oder eines Kunden. Es kennzeichnet lediglich einen erhöhten erwarteten Risikobeitrag und kann eine manuelle fachliche Prüfung priorisieren.
- Das prognostizierte **Pure Premium ist kein Kundenpreis**. Es beschreibt den modellierten erwarteten Schadenaufwand pro Exposure-Einheit. Tatsächliche Versicherungsprämien berücksichtigen zusätzliche Kosten-, Markt-, regulatorische und geschäftliche Faktoren.
- Subgruppen mit geringer Exposure oder wenigen beobachteten Schäden dürfen aufgrund höherer statistischer Unsicherheit nicht überinterpretiert werden.
- Besonders bei Severity und Pure Premium können einzelne Großschäden die beobachteten Ergebnisse kleinerer Gruppen deutlich beeinflussen.
- Subgruppenanalysen dienen der Transparenz und Risikodiagnostik. Sie stellen keinen vollständigen Nachweis von Fairness oder Nichtdiskriminierung dar.
- Merkmale wie Fahrer-Alter oder geografische Informationen sollten bei einer produktiven Nutzung zusätzlich fachlich, regulatorisch und hinsichtlich möglicher unerwünschter Benachteiligungen geprüft werden.
- Die Holdout-Ergebnisse werden ausschließlich zur finalen Diagnostik verwendet. Sie führen zu keiner nachträglichen Modellauswahl, Hyperparameter-Anpassung oder Änderung des festgelegten Train-/Test-Splits.

### Schlussfolgerung

Die Subgruppenanalyse ergänzt die globale Modellbewertung um eine wichtige Perspektive:  
**Portfolio-Kalibrierung allein reicht nicht aus, um die Zuverlässigkeit eines Modells in allen Teilpopulationen zu beurteilen.**

Für einen produktiven Einsatz sollten daher globale Modellmetriken, Subgruppen-Performance, Datenqualität und regelmäßiges Monitoring gemeinsam betrachtet werden.

In [25]:
frequency_by_age["signed_deviation"] = (
    frequency_by_age["predicted_frequency"]
    - frequency_by_age["observed_frequency"]
)

frequency_by_age["absolute_deviation"] = (
    frequency_by_age["signed_deviation"].abs()
)

frequency_by_age = frequency_by_age[
    [
        "policy_count",
        "exposure",
        "observed_claims",
        "predicted_claims",
        "observed_frequency",
        "predicted_frequency",
        "signed_deviation",
        "absolute_deviation",
        "relative_deviation",
    ]
]

frequency_by_age

,policy_count,exposure,observed_claims,predicted_claims,observed_frequency,predicted_frequency,signed_deviation,absolute_deviation,relative_deviation
driver_age_group,,,,,,,,,
18-24,5955,2424.396119,461,472.059853,0.190150,0.194712,0.004562,0.004562,0.023991
25-34,28212,12892.544473,1253,1271.291508,0.097188,0.098607,0.001419,0.001419,0.014598
35-44,34198,17522.767957,1681,1634.807587,0.095932,0.093296,-0.002636,0.002636,-0.027479
45-54,32642,17912.378251,1892,1889.971105,0.105625,0.105512,-0.000113,0.000113,-0.001072
55-64,20036,11316.824973,1017,1028.465523,0.089866,0.090879,0.001013,0.001013,0.011274
65-74,10210,6456.489457,587,617.379890,0.090916,0.095622,0.004705,0.004705,0.051754
75+,4350,3058.201412,345,304.877133,0.112811,0.099692,-0.013120,0.013120,-0.116298


In [26]:
subgroup_tables = {
    "Frequency - Age": (frequency_by_age, 135603),
    "Frequency - Region": (frequency_by_region, 135603),
    "Frequency - Area": (frequency_by_area, 135603),
    "Frequency - Density": (frequency_by_density, 135603),

    "Severity - Age": (severity_by_age, 4999),
    "Severity - Region": (severity_by_region, 4999),
    "Severity - Area": (severity_by_area, 4999),
    "Severity - Density": (severity_by_density, 4999),

    "Tweedie - Age": (tweedie_by_age, 133790),
    "Tweedie - Region": (tweedie_by_region, 133790),
    "Tweedie - Area": (tweedie_by_area, 133790),
    "Tweedie - Density": (tweedie_by_density, 133790),
}

for name, (table, expected_policies) in subgroup_tables.items():
    policy_total = table["policy_count"].sum()
    invalid_deviation = (
        table["relative_deviation"]
        .replace([float("inf"), float("-inf")], pd.NA)
        .isna()
        .sum()
    )

    print(
        f"{name}: "
        f"policies={policy_total} "
        f"(expected={expected_policies}), "
        f"invalid_relative_deviation={invalid_deviation}"
    )

Frequency - Age: policies=135603 (expected=135603), invalid_relative_deviation=0
Frequency - Region: policies=135603 (expected=135603), invalid_relative_deviation=0
Frequency - Area: policies=135603 (expected=135603), invalid_relative_deviation=0
Frequency - Density: policies=135603 (expected=135603), invalid_relative_deviation=0
Severity - Age: policies=4999 (expected=4999), invalid_relative_deviation=0
Severity - Region: policies=4999 (expected=4999), invalid_relative_deviation=0
Severity - Area: policies=4999 (expected=4999), invalid_relative_deviation=0
Severity - Density: policies=4999 (expected=4999), invalid_relative_deviation=0
Tweedie - Age: policies=133790 (expected=133790), invalid_relative_deviation=0
Tweedie - Region: policies=133790 (expected=133790), invalid_relative_deviation=0
Tweedie - Area: policies=133790 (expected=133790), invalid_relative_deviation=0
Tweedie - Density: policies=133790 (expected=133790), invalid_relative_deviation=0
